BREAKDOWN EHRSHOT CSV

In [ ]:
import pandas as pd
import csv

In [ ]:
def split_csv_by_rows(input_file, rows_per_file):
    with open(input_file, 'r', newline='', encoding='utf-8') as infile:
        reader = csv.reader(infile)
        header = next(reader)  # read the header

        file_count = 1
        rows = []

        for i, row in enumerate(reader, start=1):
            rows.append(row)
            if i % rows_per_file == 0:
                output_file = f'EHRSHOT_ASSETS/data/output_{file_count}.csv'
                with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
                    writer = csv.writer(outfile)
                    writer.writerow(header)
                    writer.writerows(rows)
                print(f'Written {output_file}')
                rows = []
                file_count += 1

        # Write any remaining rows
        if rows:
            output_file = f'EHRSHOT_ASSETS/data/output_{file_count}.csv'
            with open(output_file, 'w', newline='', encoding='utf-8') as outfile:
                writer = csv.writer(outfile)
                writer.writerow(header)
                writer.writerows(rows)
            print(f'Written {output_file}')

In [ ]:
CHUNK_SIZE = 1000000

In [ ]:
split_csv_by_rows('EHRSHOT_ASSETS/data/ehrshot.csv', CHUNK_SIZE)

In [ ]:
for i in range(0, 34):
    fname = f'EHRSHOT_ASSETS/data/output_{i}.csv'
    df = pd.read_csv(fname)
    df = df.sort_values(['patient_id', 'start'])
    df.to_parquet(fname[:-4]+'.parquet', index=False)

In [ ]:
N = 200
pat_ids = []
for i in range(0, 35):
    start = i*N
    end = min((i+1)*N, 6739)
    pat_id = pat_ids[start:end]

    df_i = None
    for j in range(1, 43):
        fname = f'EHRSHOT_ASSETS/data/output_{j}.csv'
        df = pd.read_csv(fname)
        df = df[df['patient_id'].isin(pat_id)]
        if df_i is None:
            df_i = df
        else:
            df_i = pd.concat([df_i, df])
    df_i.to_csv(f'data/output_{i}.csv')

LABS

In [1]:
import os
import pandas as pd
import numpy as np
from scipy.stats import linregress, kurtosis, skew

In [2]:
DATA_DIR = f'/home/seewon/ehrshot/data/'

In [3]:
def get_labs(phenotype):
    df_phenotype = None

    label_df = pd.read_csv(os.path.join(DATA_DIR, f'benchmark/{phenotype}/labeled_patients.csv'))
    ids = label_df['patient_id'].unique()

    for i in range(0, 34):
        df = pd.read_parquet(os.path.join(DATA_DIR, f'data/output_{i}.parquet'))
        df = df[df['patient_id'].isin(ids)]

        if len(df) == 0: continue
        elif df_phenotype is None:
            df_phenotype = df
        else: df_phenotype = pd.concat([df_phenotype, df])
    return df_phenotype

ICU STAY

In [ ]:
df_icu = get_labs('guo_icu')
df_icu.to_parquet(os.path.join(DATA_DIR, 'benchmark/guo_icu/icu.parquet'))

In [ ]:
label_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_patients.csv'))
label_df = label_df.drop_duplicates()

In [ ]:
pos_df = label_df[label_df['value']==True]
pos_df = pos_df.groupby('patient_id').min().reset_index()

In [ ]:
neg_df = label_df[(label_df['value']==False) & (~label_df['patient_id'].isin(pos_df['patient_id']))]
neg_df = neg_df.groupby('patient_id').min().reset_index()

In [ ]:
label_df = pd.concat([pos_df, neg_df], ignore_index=True)

In [ ]:
label_df.to_parquet(os.path.join(DATA_DIR, f'{TASK}_label.parquet'))

CHEST X-RAY

In [ ]:
df_chest = get_labs('chexpert')
df_chest.to_parquet(os.path.join(DATA_DIR, 'benchmark/chexpert/chexpert.parquet'))

In [ ]:
label_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_patients.csv'))
label_df = label_df.drop_duplicates()

In [ ]:
def int_to_binary_label_list(label_int, length=14):
    binary_str = format(label_int, f'0{length}b')[::-1]
    return [int(bit) for bit in binary_str]

In [ ]:
CHEXPERT_LABELS = [
    "No Finding",
    "Enlarged Cardiomediastinum",
    "Cardiomegaly",
    "Lung Lesion",
    "Lung Opacity",
    "Edema",
    "Consolidation",
    "Pneumonia",
    "Atelectasis",
    "Pneumothorax",
    "Pleural Effusion",
    "Pleural Other",
    "Fracture",
    "Support Devices",
]

In [ ]:
values = label_df.groupby('value')['patient_id'].nunique().sort_values().reset_index()['value'].unique()

In [ ]:
label_df['priority'] = label_df['value'].map({v: i for i, v in enumerate(values)})
label_df = label_df.sort_values(by=['patient_id', 'priority']).drop_duplicates('patient_id')
label_df = label_df.drop(columns='priority')

In [ ]:
label_df['value'] = label_df['value'].apply(int_to_binary_label_list)

In [ ]:
label_df.to_parquet(os.path.join(DATA_DIR, f'{TASK}_label.parquet'))

THROMBOCYTOPENIA

In [ ]:
THROMBOCYTOPENIA = ['LOINC/LP393218-5', 'LOINC/LG32892-8', 'LOINC/777-3']

In [ ]:
df_thrombocytopenia = get_labs('lab_thrombocytopenia')
df_thrombocytopenia.to_parquet(os.path.join(DATA_DIR, 'thrombocytopenia.parquet'))

HYPERGLYCEMIA

In [ ]:
HYPERGLYCEMIA = ['SNOMED/33747003', 'LOINC/LP416145-3', 'LOINC/14749-6']

In [ ]:
df_hypoglycemia = get_labs('lab_hypoglycemia')
df_hypoglycemia.to_parquet(os.path.join(DATA_DIR, 'hypoglycemia.parquet'))

HYPONATREMIA

In [ ]:
HYPONATREMIA = ['LOINC/LG11363-5', 'LOINC/2951-2', 'LOINC/2947-0']

In [ ]:
df_hyponatremia = get_labs('lab_hyponatremia')
df_hyponatremia.to_parquet(os.path.join(DATA_DIR, 'hyponatremia.parquet'))

ANEMIA

In [3]:
ANEMIA = ['LOINC/LP392452-1']

In [ ]:
df_anemia = get_labs('lab_anemia')
df_anemia.to_parquet(os.path.join(DATA_DIR, 'anemia.parquet'))

HYPERKALEMIA

In [ ]:
HYPERKALEMIA = ['LOINC/LG7931-1', 'LOINC/LP386618-5', 'LOINC/LG10990-6', 'LOINC/6298-4', 'LOINC/2823-3']

In [ ]:
df_hyperkalemia = get_labs('lab_hyperkalemia')
df_hyperkalemia.to_parquet(os.path.join(DATA_DIR, 'hyperkalemia.parquet'))

LABELS: LABS

In [ ]:
ROOT_DIR = '/home/seewon/ehrshot/data'
TASK_PREFIX = 'guo_' # lab_i, guo_
TASK = 'icu' # 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia', 'anemia' 'chexpert
DATA_DIR = os.path.join(ROOT_DIR, f'{TASK_PREFIX}{TASK}')

In [ ]:
label_df = pd.read_csv(os.path.join(DATA_DIR, 'labeled_patients.csv'))
label_df = label_df.drop_duplicates()

In [ ]:
three_df = label_df[label_df['value']==3]
two_df = label_df[(label_df['value']==2) & (~label_df['patient_id'].isin(three_df['patient_id']))]
one_df = label_df[(label_df['value']==1) & (~label_df['patient_id'].isin(three_df['patient_id'])) & (~label_df['patient_id'].isin(two_df['patient_id']))]
neg_df = label_df[(label_df['value']==0) & (~label_df['patient_id'].isin(three_df['patient_id'])) 
                  & (~label_df['patient_id'].isin(two_df['patient_id'])) & (~label_df['patient_id'].isin(one_df['patient_id']))]

In [ ]:
three_df = three_df.groupby('patient_id').min().reset_index()
two_df = two_df.groupby('patient_id').min().reset_index()
one_df = one_df.groupby('patient_id').min().reset_index()
neg_df = neg_df.groupby('patient_id').min().reset_index()

In [ ]:
label_df = pd.concat([three_df, two_df, one_df, neg_df], ignore_index=True)

In [ ]:
label_df.to_parquet(os.path.join(DATA_DIR, f'{TASK}_label.parquet'))

DEMOGRAPHICS

In [3]:
task_type = 'guo_icu'
task_name = 'icu'

In [8]:
task_name = 'thrombocytopenia'      # anemia, hyperkalemia, hypoglycemia, hyponatremia
task_type = f'lab_{task_name}'

In [9]:
df = pd.read_parquet(f'data/{task_type}/{task_name}.parquet')
label_df = pd.read_parquet(f'data/{task_type}//{task_name}_label.parquet')

In [ ]:
assert(df['patient_id'].nunique() == len(label_df))

True

In [10]:
demo_df = df[df['omop_table'] == 'person']
df = df[~(df['omop_table'] == 'person')]

In [21]:
start_time = df[['patient_id', 'start']].groupby('patient_id').min().reset_index()
label_df = label_df.merge(start_time, on='patient_id')

In [22]:
birth_df = demo_df[demo_df['code'] == 'SNOMED/3950001'][['patient_id', 'start']]
birth_df = birth_df.rename(columns={'start': 'birth'})
label_df = label_df.merge(birth_df, on='patient_id', how='left')

In [23]:
label_df['start'] = pd.to_datetime(label_df['start'])
label_df['birth'] = pd.to_datetime(label_df['birth'])
label_df['age'] = label_df.apply(lambda x: x['start'].year - x['birth'].year - ((x['start'].month, x['start'].day) < (x['birth'].month, x['birth'].day)), axis=1)
label_df['prediction_time'] = pd.to_datetime(label_df['prediction_time'])

In [24]:
gender_df = demo_df[demo_df['code'].str.startswith('Gender/')][['patient_id', 'code']]
gender_df = gender_df.rename(columns={'code' : 'gender'})
label_df = label_df.merge(gender_df, on='patient_id', how='left')

In [25]:
race_df = demo_df[demo_df['code'].str.startswith('Race/')][['patient_id', 'code']]
race_df = race_df.rename(columns={'code' : 'race'})
label_df = label_df.merge(race_df, on='patient_id', how='left')
label_df['race'] = label_df['race'].fillna('Race/0')

In [26]:
ethnicity_df = demo_df[demo_df['code'].str.startswith('Ethnicity/')][['patient_id', 'code']]
ethnicity_df = ethnicity_df.rename(columns={'code' : 'ethnicity'})
label_df = label_df.merge(ethnicity_df, on='patient_id', how='left')
label_df['ethnicity'] = label_df['ethnicity'].fillna('Ethnicity/None')

In [27]:
label_df['race'] = label_df['race'].apply(lambda x: x.split('/', 1)[1])
label_df['gender'] = label_df['gender'].apply(lambda x: x.split('/', 1)[1])
label_df['ethnicity'] = label_df['ethnicity'].apply(lambda x: x.split('/', 1)[1])

In [28]:
label_df = label_df.drop(columns=['label_type'])

In [29]:
df['patient_id'].nunique() == len(label_df)

True

In [30]:
label_df.to_parquet(f'data/{task_type}/{task_name}_label.parquet')

In [ ]:
df.to_parquet(f'data/{task_type}/{task_name}.parquet')

SPLIT LABEL AND DEMOGRAPHICS

In [3]:
task_name = 'icu'
task_type = f'guo_{task_name}'
label_df = pd.read_parquet(f'data/{task_type}/{task_name}_label.parquet')

In [ ]:
demo_df = label_df[['patient_id', 'gender', 'race', 'ethnicity', 'age', 'birth']]
label_df = label_df[['patient_id', 'prediction_time', 'value', 'start']]

In [ ]:
label_df.to_parquet(f'data/{task_type}/{task_name}_label.parquet')
demo_df.to_parquet(f'data/{task_type}/{task_name}_demo.parquet')

In [10]:
for task_name in ['anemia', 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia']:
    task_type = f'lab_{task_name}'
    label_df = pd.read_parquet(f'data/{task_type}/{task_name}_label.parquet')

    demo_df = label_df[['patient_id', 'gender', 'race', 'ethnicity', 'age', 'birth']]
    label_df = label_df[['patient_id', 'prediction_time', 'value', 'start']]
    
    label_df.to_parquet(f'data/{task_type}/{task_name}_label.parquet')
    demo_df.to_parquet(f'data/{task_type}/{task_name}_demo.parquet')

In [16]:
for task_name in ['anemia', 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia', 'icu']:
    if task_name == 'icu': task_type = f'guo_{task_name}'
    else: task_type = f'lab_{task_name}'
    demo_df = pd.read_parquet(f'data/{task_type}/{task_name}_demo.parquet')

    demo_df['gender_ind'] = pd.factorize(demo_df['gender'])[0]
    demo_df['ethnicity_ind'] = pd.factorize(demo_df['ethnicity'])[0]
    demo_df['race_ind'] = pd.factorize(demo_df['race'])[0]

    demo_df.to_parquet(f'data/{task_type}/{task_name}_demo.parquet')

In [ ]:
for task_name in ['anemia', 'hyperkalemia', 'hypoglycemia', 'hyponatremia', 'thrombocytopenia']:
    task_type = f'lab_{task_name}'
    label_df = pd.read_parquet(f'data/{task_type}/{task_name}_label.parquet')

    label_df['value_multi'] = label_df['value']
    label_df['value']  = label_df['value'] == 3
    label_df.to_parquet(f'data/{task_type}/{task_name}_label.parquet')

For SAMPLE TIME, first split into VISITS

In [ ]:
TASK = 'hypoglycemia'

In [ ]:
label_df = pd.read_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}_label.parquet')

In [ ]:
df = pd.read_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}.parquet')
df['start'] = pd.to_datetime(df['start'])
df = df.sort_values(['patient_id', 'start'])

In [ ]:
def assign_group_numbers(group):
    time_diff = group['start'].diff().fillna(pd.Timedelta(days=0))
    new_group = (time_diff >= pd.Timedelta(days=1)).cumsum()
    return new_group + 1

In [ ]:
df['N'] = df.groupby('patient_id', group_keys=False).apply(assign_group_numbers).reset_index(drop=True)

In [ ]:
pred_df = df[['patient_id', 'start', 'N']].merge(label_df[['patient_id', 'prediction_time']], on='patient_id')
pred_df = pred_df[(pred_df['start'] - pred_df['prediction_time']).abs() < pd.Timedelta(hours=24)]
pred_df = pred_df.drop(columns=['start', 'prediction_time']).drop_duplicates()

In [ ]:
label_df = label_df.merge(pred_df.groupby('patient_id').min().reset_index(), on='patient_id')
label_df = label_df.rename(columns={'N': 'N_min'})
label_df = label_df.merge(pred_df.groupby('patient_id').max().reset_index(), on='patient_id')
label_df = label_df.rename(columns={'N': 'N_max'})

In [ ]:
df = df.merge(label_df[['patient_id', 'N_min', 'N_max']], on='patient_id')

In [ ]:
df = df[df['N'] >= df['N_min']]
df = df[df['N'] <= df['N_max']]
df = df.drop(columns=['N', 'N_min', 'N_max'])

In [ ]:
df.to_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}2.parquet')

THEN, SAMPLE TIME

In [ ]:
label_df = label_df.drop(columns=['start', 'N_min', 'N_max'])

In [ ]:
start_df = df.groupby('patient_id')['start'].min().reset_index()
label_df = label_df.merge(start_df, on='patient_id', how='left')

In [ ]:
pos_df = label_df[label_df['value']]
neg_df = label_df[~label_df['value']]

In [ ]:
pos_df = pos_df[(pos_df['prediction_time']  - pos_df['start']) >= pd.Timedelta(hours=1)]

In [ ]:
neg_df = neg_df.drop(columns=['prediction_time'])

In [ ]:
delta = (pos_df['prediction_time'] - pos_df['start']).dt.total_seconds() / 3600
times = delta.sample(n=int(len(neg_df)*1.2), replace=True) + np.random.normal(0, 0.5, int(len(neg_df)*1.2))
neg_df['Sample_time'] = [t for t in times if t > 1.5][:len(neg_df)]
neg_df['Sample_time'] = neg_df['Sample_time'].apply(lambda x: pd.Timedelta(hours=x)) + neg_df['start']

In [ ]:
label_df = pd.concat([pos_df, neg_df], ignore_index=True)

In [ ]:
label_df.to_parquet(f'/home/seewon/ehrshot/data/lab_{TASK}/{TASK}_label2.parquet')

VISITS?

In [ ]:
df = pd.read_parquet(os.path.join(DATA_DIR, f'{TASK}.parquet'))

In [ ]:
visit_detail = df[df['omop_tablae']=='visit_detail']
visit_duration = visit_detail.groupby('visit_id')['start'].min().reset_index()
visit_duration = visit_duration.merge(visit_detail.groupby('visit_id')['end'].max().reset_index(), on='visit_id')

In [ ]:
visit_occ = df[(df['omop_table']=='visit_occurrence')]
visit_duration2 = visit_occ.groupby('visit_id')['start'].min().reset_index()
visit_duration2 = visit_duration2.merge(visit_occ.groupby('visit_id')['end'].max().reset_index(), on='visit_id')

In [ ]:
visit_duration = pd.concat([visit_duration, visit_duration2], ignore_index=True)

In [ ]:
visits_df = visit_duration.groupby('visit_id')['start'].min().reset_index()
visits_df = visits_df.merge(visit_duration.groupby('visit_id')['end'].max().reset_index(), on='visit_id')
visits_df = visits_df.merge(df[['patient_id', 'visit_id']].drop_duplicates(), on='visit_id', how='left')

In [ ]:
visits_df.to_parquet(os.path.join(ROOT_DIR, 'visits.parquet'))

In [2]:
df = pd.read_parquet('/home/seewon/ehrshot/data/lab_hyperkalemia/hyperkalemia2.parquet')

In [3]:
HYPERKALEMIA = ['LOINC/LG7931-1', 'LOINC/LP386618-5', 'LOINC/LG10990-6', 'LOINC/6298-4', 'LOINC/2823-3']

In [10]:
pd.to_numeric(df[df['code'].isin(HYPERKALEMIA)]['value']).max()

18.299999237060547